In [1]:
import os
import numpy as np
import pandas as pd
from ogcore.utils import safe_read_pickle
from ogcore.output_tables import dynamic_revenue_decomposition

In [2]:
CUR_DIR = './'
base_dir = os.path.join(CUR_DIR, 'Current_Law', "OUTPUT")
reform_dir = os.path.join(CUR_DIR, 'TCJA_Ext_Plus_Prod_Gain_1.02', "OUTPUT")

base_tpi = safe_read_pickle(os.path.join(base_dir, "TPI", "TPI_vars.pkl"))
base_params = safe_read_pickle(os.path.join(base_dir, "model_params.pkl"))
base_ss = safe_read_pickle(os.path.join(base_dir, "SS", "SS_vars.pkl"))
reform_tpi = safe_read_pickle(os.path.join(reform_dir, "TPI", "TPI_vars.pkl"))
reform_params = safe_read_pickle(os.path.join(reform_dir, "model_params.pkl"))
reform_ss = safe_read_pickle(os.path.join(reform_dir, "SS", "SS_vars.pkl"))

In [3]:
df = dynamic_revenue_decomposition(base_params, base_tpi, base_ss, reform_params, reform_tpi, reform_ss, start_year=2025, num_years=10, full_break_out=True)
df

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034,SS
0,IIT: Pct Change due to tax rates,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44,-5.44
1,IIT: Pct Change due to behavior,-0.77,-0.52,-0.09,0.23,0.57,0.94,1.37,1.87,2.48,3.28,0.93,4.01
2,IIT: Pct Change due to macro,2.61,5.22,7.94,10.76,13.72,16.81,20.02,23.36,26.82,30.41,15.85,28.95
3,IIT: Overall Pct Change in taxes,-3.72,-1.03,1.97,4.98,8.15,11.49,15.04,18.82,22.90,27.36,10.56,26.83
4,CIT: Pct Change due to tax rates,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
5,CIT: Pct Change due to behavior,8.46,17.80,27.96,38.43,49.46,61.06,73.27,86.13,99.76,114.31,57.65,113.24
6,CIT: Pct Change due to macro,-5.83,-10.31,-14.47,-18.20,-21.64,-24.80,-27.70,-30.36,-32.80,-35.05,-24.07,-37.00
7,CIT: Overall Pct Change in taxes,2.13,5.66,9.44,13.23,17.11,21.11,25.27,29.63,34.24,39.19,19.70,34.34
8,All: Pct Change due to tax rates,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.13,-5.12
9,All: Pct Change due to behavior,-0.21,0.59,1.61,2.54,3.53,4.59,5.74,6.99,8.41,10.04,4.37,10.75


In [4]:
# Now apply these percentage changes to the baseline revenue
# Take CBO baseline (to include not just IIT)
# Taken from CBO June 2024 Budgdet Outlook, 2026-2034
base_revenue = np.array([5.038, 5.394, 5.756, 5.944, 6.133, 6.354, 6.661, 6.899, 7.176, 7.459])
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[8:, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
8,Rev Change Due to Tax Rates,-0.26,-0.28,-0.30,-0.30,-0.31,-0.33,-0.34,-0.35,-0.37,-0.38,-3.22
9,Rev Change Due to Behavior,-0.01,0.03,0.09,0.15,0.22,0.29,0.38,0.48,0.60,0.75,2.99
10,Rev Change Due to Macro,0.10,0.22,0.36,0.50,0.65,0.82,1.02,1.22,1.44,1.69,8.03
11,Total Revenue Change,-0.17,-0.03,0.14,0.32,0.53,0.77,1.04,1.34,1.69,2.09,7.72


In [5]:
# Get level changes just for IIT + Payroll
# Taken from CBO June 2024 Budgdet Outlook, 2026-2034
base_revenue = np.array([4.287, 4.655, 5.007, 5.184, 5.365, 5.573, 5.783, 5.994, 6.227, 6.476])
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[0:3, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,-0.23,-0.25,-0.27,-0.28,-0.29,-0.30,-0.31,-0.33,-0.34,-0.35,-2.97
1,Rev Change Due to Behavior,-0.03,-0.02,-0.00,0.01,0.03,0.05,0.08,0.11,0.15,0.21,0.59
2,Rev Change Due to Macro,0.11,0.24,0.40,0.56,0.74,0.94,1.16,1.40,1.67,1.97,9.18
3,Total Revenue Change,-0.16,-0.05,0.10,0.26,0.44,0.64,0.87,1.13,1.43,1.77,6.42


In [6]:
result_df_static = pd.read_csv('../../Tax-Calculator-thru74/tax_brain_result_wo_behresp.csv', index_col = 0)

In [7]:
# Or we can use the Tax-Calc baseline for a direct comparison
base_revenue = result_df_static.loc["Base", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
# Above is just over all revenue so only apply to those rows
df_levels = df.loc[0:3, df.columns[:-2]]
df_levels.loc[:, df_levels.columns[1:]] =  df_levels.loc[:, df_levels.columns[1:]] * np.tile(base_revenue.reshape(1, 10), (df_levels.shape[0], 1)) / 100
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,-0.22,-0.25,-0.26,-0.27,-0.28,-0.29,-0.30,-0.32,-0.33,-0.34,-2.86
1,Rev Change Due to Behavior,-0.03,-0.02,-0.00,0.01,0.03,0.05,0.08,0.11,0.15,0.21,0.57
2,Rev Change Due to Macro,0.11,0.24,0.38,0.53,0.71,0.90,1.12,1.36,1.62,1.91,8.87
3,Total Revenue Change,-0.15,-0.05,0.09,0.25,0.42,0.62,0.84,1.09,1.38,1.72,6.22


In [8]:
# jason's get-around

df_levels = df.loc[0:3, df.columns[:-2]]
tc_diff = result_df_static.loc["Difference", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
tc_reform = result_df_static.loc["Reform", ['2025', '2026', '2027', '2028', '2029', '2030', '2031', '2032', '2033', '2034']].values
df_levels.loc[0, df_levels.columns[1:]] = tc_diff
df_levels.loc[1, df_levels.columns[1:]] = (df.loc[1, df_levels.columns[1:]] / 100) * tc_reform
df_levels.loc[2, df_levels.columns[1:]] = (df.loc[2, df_levels.columns[1:]] / 100) * df_levels.loc[1, df_levels.columns[1:]]
df_levels.loc[3, df_levels.columns[1:]] = df_levels.loc[0, df_levels.columns[1:]] + df_levels.loc[1, df_levels.columns[1:]] + df_levels.loc[2, df_levels.columns[1:]]
df_levels["2025-2034"] = df_levels.loc[:, df_levels.columns[1:]].sum(axis=1)
df_levels.Variable = ["Rev Change Due to Tax Rates", "Rev Change Due to Behavior", "Rev Change Due to Macro", "Total Revenue Change"]
df_levels

Year,Variable,2025,2026,2027,2028,2029,2030,2031,2032,2033,2034,2025-2034
0,Rev Change Due to Tax Rates,0.00,-0.29,-0.30,-0.31,-0.32,-0.33,-0.34,-0.35,-0.36,-0.36,-2.95
1,Rev Change Due to Behavior,-0.03,-0.02,-0.00,0.01,0.03,0.05,0.07,0.10,0.14,0.19,0.54
2,Rev Change Due to Macro,-0.00,-0.00,-0.00,0.00,0.00,0.01,0.01,0.02,0.04,0.06,0.15
3,Total Revenue Change,-0.03,-0.31,-0.30,-0.30,-0.29,-0.27,-0.25,-0.22,-0.18,-0.11,-2.26


In [9]:
df_levels.to_csv('og_usa_result_w_tcja_prod_2.csv')